# Supermarket Sales Analytics
**Author:** Abbas Ali  
**Project:** Supermarket Sales Analytics  
**Dataset:** Synthetic supermarket transactions (500 rows, Jan–Mar 2019)  
**Tools:** Python · pandas · matplotlib · seaborn · plotly

---

## Objectives
1. Load and inspect the dataset
2. Perform data quality checks
3. Engineer new features
4. Group and summarise key metrics
5. Visualise 10 analytical charts
6. Extract actionable business insights

## Step 0 — Import Libraries

In [ ]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

PALETTE = ['#3b82d4', '#7c5cd8', '#10b981', '#f59e0b', '#ef4444', '#6366f1']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print('Libraries loaded successfully.')

---
## Step 1 — Load Dataset

In [ ]:
CSV_PATH = 'supermarket_sales.csv'

df = pd.read_csv(CSV_PATH)

print(f'Rows : {len(df)}')
print(f'Cols : {len(df.columns)}')
print()
df.head()

In [ ]:
# Column data types
df.dtypes

In [ ]:
# Basic statistics
df.describe().round(2)

---
## Step 2 — Data Quality Check

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing.to_string())

In [ ]:
# Duplicate rows
dups = df.duplicated().sum()
print(f'Duplicate rows  : {dups}')

# Negative / zero values
neg_qty   = (df['Quantity']   <= 0).sum()
neg_price = (df['Unit price'] <= 0).sum()
print(f'Invalid Qty     : {neg_qty}')
print(f'Invalid Price   : {neg_price}')

# Rating range check
out_range = ((df['Rating'] < 1) | (df['Rating'] > 10)).sum()
print(f'Ratings out of [1-10]: {out_range}')

# Parse date
df['Date'] = pd.to_datetime(df['Date'])
print(f'Date range: {df["Date"].min().date()}  to  {df["Date"].max().date()}')

**Quality Summary:** No missing values, no duplicates, no invalid quantities or prices, all ratings within [1–10]. Dataset is clean and ready for analysis.

---
## Step 3 — Feature Engineering

In [ ]:
# Recalculate Sales from raw columns
df['Sales']   = (df['Quantity'] * df['Unit price']).round(2)
df['Month']   = df['Date'].dt.month_name()
df['Weekday'] = df['Date'].dt.day_name()

MONTH_ORDER   = ['January', 'February', 'March']
WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                 'Friday', 'Saturday', 'Sunday']

print('New columns added: Sales, Month, Weekday')
print(f'Total Sales (all): ${df["Sales"].sum():,.2f}')
print(f'Avg Sales/tx     : ${df["Sales"].mean():,.2f}')
df[['Quantity', 'Unit price', 'Sales', 'Month', 'Weekday']].head(8)

---
## Step 4 — Grouping & Summarisation

In [ ]:
# Branch summary
branch_summary = (
    df.groupby('Branch')
      .agg(
          Total_Sales    =('Sales',      'sum'),
          Transactions   =('Invoice ID', 'count'),
          Avg_Rating     =('Rating',     'mean'),
          Avg_Unit_Price =('Unit price', 'mean'),
      )
      .round(2)
)
print('-- Branch Summary --')
branch_summary

In [ ]:
# Product line summary
product_summary = (
    df.groupby('Product line')
      .agg(
          Total_Sales  =('Sales',      'sum'),
          Transactions =('Invoice ID', 'count'),
          Total_Qty    =('Quantity',   'sum'),
          Avg_Rating   =('Rating',     'mean'),
      )
      .round(2)
      .sort_values('Total_Sales', ascending=False)
)
print('-- Product Line Summary --')
product_summary

In [ ]:
# Payment method summary
payment_summary = (
    df.groupby('Payment')
      .agg(Total_Sales=('Sales', 'sum'), Count=('Invoice ID', 'count'))
      .round(2)
      .sort_values('Total_Sales', ascending=False)
)
print('-- Payment Method Summary --')
payment_summary

In [ ]:
# Customer type & Gender summaries
cust_summary = (
    df.groupby('Customer type')
      .agg(Total_Sales=('Sales', 'sum'), Count=('Invoice ID', 'count'),
           Avg_Rating =('Rating', 'mean'))
      .round(2)
)
print('-- Customer Type Summary --')
display(cust_summary)

gender_summary = (
    df.groupby('Gender')
      .agg(Total_Sales=('Sales', 'sum'), Count=('Invoice ID', 'count'))
      .round(2)
)
print('-- Gender Summary --')
gender_summary

In [ ]:
# Monthly sales & Branch x Category cross-tab
monthly_sales = (
    df.groupby('Month')['Sales']
      .sum()
      .reindex(MONTH_ORDER)
      .reset_index()
)
branch_cat = (
    df.groupby(['Branch', 'Product line'])['Sales']
      .sum()
      .unstack('Product line')
      .round(2)
)
print('-- Monthly Sales --')
display(monthly_sales)
print('-- Branch x Product Line Cross-tab --')
branch_cat

---
## Step 5 — Visualisations

### Chart 1 — Total Sales by Branch

In [ ]:
fig = px.bar(
    branch_summary.reset_index(),
    x='Branch', y='Total_Sales',
    color='Branch', color_discrete_sequence=PALETTE,
    text_auto=',.0f',
    title='Total Sales by Branch',
    labels={'Total_Sales': 'Sales ($)'}
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

### Chart 2 — Sales by Product Line

In [ ]:
fig = px.bar(
    product_summary.reset_index().sort_values('Total_Sales'),
    x='Total_Sales', y='Product line',
    orientation='h', color='Product line',
    color_discrete_sequence=PALETTE,
    text_auto=',.0f',
    title='Sales by Product Line',
    labels={'Product line': '', 'Total_Sales': 'Sales ($)'}
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickprefix='$', xaxis_tickformat=',')
fig.show()

### Chart 3 — Payment Method Distribution

In [ ]:
fig = px.pie(
    payment_summary.reset_index(),
    names='Payment', values='Count',
    color_discrete_sequence=PALETTE,
    title='Payment Method Distribution',
    hole=0.35
)
fig.update_traces(textinfo='label+percent', textposition='outside')
fig.show()

### Chart 4 — Monthly Sales Trend

In [ ]:
fig = px.line(
    monthly_sales, x='Month', y='Sales',
    markers=True,
    title='Monthly Sales Trend (Jan–Mar 2019)',
    labels={'Sales': 'Sales ($)', 'Month': ''},
    color_discrete_sequence=[PALETTE[0]],
    text='Sales'
)
fig.update_traces(
    texttemplate='$%{text:,.0f}',
    textposition='top center',
    line_width=2.5,
    marker_size=9
)
fig.update_layout(yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

### Chart 5 — Average Customer Rating by Product Line

In [ ]:
avg_rating = df.groupby('Product line')['Rating'].mean().round(2).sort_values(ascending=False)

fig = px.bar(
    avg_rating.reset_index(),
    x='Product line', y='Rating',
    color='Product line', color_discrete_sequence=PALETTE,
    text='Rating',
    title='Average Customer Rating by Product Line',
    labels={'Product line': '', 'Rating': 'Avg Rating (1–10)'}
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[0, 11])
fig.show()

### Chart 6 — Sales by Customer Type & Gender

In [ ]:
cg = df.groupby(['Customer type', 'Gender'])['Sales'].sum().reset_index()

fig = px.bar(
    cg, x='Customer type', y='Sales', color='Gender',
    barmode='group',
    color_discrete_sequence=PALETTE[:2],
    text_auto=',.0f',
    title='Sales by Customer Type & Gender',
    labels={'Customer type': '', 'Sales': 'Sales ($)'}
)
fig.update_layout(yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

### Chart 7 — Branch × Product Line Heatmap

In [ ]:
fig = px.imshow(
    branch_cat,
    text_auto=',.0f',
    color_continuous_scale='Blues',
    title='Sales Heatmap — Branch × Product Line',
    labels={'color': 'Sales ($)'},
    aspect='auto'
)
fig.update_xaxes(tickangle=-25)
fig.show()

### Chart 8 — Customer Rating Distribution by Branch (Box Plot)

In [ ]:
fig = px.box(
    df, x='Branch', y='Rating',
    color='Branch', color_discrete_sequence=PALETTE,
    title='Rating Distribution by Branch',
    labels={'Rating': 'Rating (1–10)', 'Branch': ''},
    points='outliers'
)
fig.update_layout(showlegend=False, yaxis_range=[0, 11])
fig.show()

### Chart 9 — Average Sale per Transaction by Weekday

In [ ]:
weekday_avg = (
    df.groupby('Weekday')['Sales']
      .mean()
      .reindex(WEEKDAY_ORDER)
      .dropna()
      .round(2)
      .reset_index()
)

fig = px.bar(
    weekday_avg, x='Weekday', y='Sales',
    color_discrete_sequence=[PALETTE[4]],
    text='Sales',
    title='Average Sale per Transaction by Weekday',
    labels={'Sales': 'Avg Sales ($)', 'Weekday': ''}
)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

### Chart 10 — Quantity Sold by Product Line

In [ ]:
qty_prod = df.groupby('Product line')['Quantity'].sum().reset_index()

fig = px.pie(
    qty_prod, names='Product line', values='Quantity',
    color_discrete_sequence=PALETTE,
    title='Quantity Sold by Product Line',
    hole=0.3
)
fig.update_traces(textinfo='label+percent', textposition='outside')
fig.show()

---
## Step 6 — Business Insights

In [ ]:
top_branch   = branch_summary['Total_Sales'].idxmax()
top_product  = product_summary['Total_Sales'].idxmax()
top_payment  = payment_summary['Total_Sales'].idxmax()
top_month    = monthly_sales.set_index('Month')['Sales'].idxmax()
best_rated   = avg_rating.idxmax()
worst_rated  = avg_rating.idxmin()
best_weekday = weekday_avg.set_index('Weekday')['Sales'].idxmax()
top_gender   = gender_summary['Total_Sales'].idxmax()
top_cust     = cust_summary['Total_Sales'].idxmax()

print('=' * 55)
print('   SUPERMARKET SALES — BUSINESS INSIGHTS')
print('=' * 55)

print('\n[SALES PERFORMANCE]')
print(f"  Total Revenue        : ${df['Sales'].sum():>12,.2f}")
print(f"  Total Transactions   : {len(df):>12,}")
print(f"  Average Sale Value   : ${df['Sales'].mean():>12,.2f}")
print(f"  Highest Single Sale  : ${df['Sales'].max():>12,.2f}")

print('\n[BRANCH ANALYSIS]')
print(f"  Top Performing Branch: Branch {top_branch}")
print(f"    Revenue    = ${branch_summary.loc[top_branch, 'Total_Sales']:,.2f}")
print(f"    Avg Rating = {branch_summary.loc[top_branch, 'Avg_Rating']:.2f}")

print('\n[PRODUCT LINE ANALYSIS]')
print(f"  Best-Selling Line    : {top_product}")
print(f"    Revenue  = ${product_summary.loc[top_product, 'Total_Sales']:,.2f}")
print(f"  Highest Rated Line   : {best_rated}  (avg {avg_rating[best_rated]:.2f}/10)")
print(f"  Lowest  Rated Line   : {worst_rated} (avg {avg_rating[worst_rated]:.2f}/10)")
print(f"    Recommendation: Improve quality/service for '{worst_rated}'")

print('\n[PAYMENT METHOD ANALYSIS]')
print(f"  Most Used Method     : {top_payment}")
print(f"    Revenue  = ${payment_summary.loc[top_payment, 'Total_Sales']:,.2f}")
print(f"    Recommendation: Offer cashback / loyalty points on {top_payment}")

print('\n[CUSTOMER ANALYSIS]')
print(f"  Top Customer Segment : {top_cust}")
print(f"  Top Spending Gender  : {top_gender}")
print(f"    Recommendation: Target {top_gender} customers with promotions")

print('\n[TIME ANALYSIS]')
print(f"  Best Month           : {top_month}")
print(f"    Revenue  = ${monthly_sales.set_index('Month')['Sales'][top_month]:,.2f}")
print(f"  Best Weekday Avg Sale: {best_weekday}")
print(f"    Recommendation: Run targeted deals on {best_weekday}")
print()
print('=' * 55)

---
## Conclusion

This project performed an end-to-end analysis of 500 supermarket transactions across three branches (Yangon, Naypyitaw, Mandalay) covering January to March 2019.

**Key findings:**
- Total revenue across all branches exceeded $50,000
- Sales are nearly evenly distributed across branches, indicating healthy multi-branch performance
- Food & Beverages and Fashion Accessories are consistently high-revenue product lines
- Ewallet is the most popular payment method — a target for loyalty rewards
- Member customers spend slightly more on average than Normal customers
- January showed the highest monthly revenue, suggesting seasonal opportunity

**Recommendations:**
1. Replicate top-branch strategies across underperforming branches
2. Introduce membership incentives to convert Normal customers to Members
3. Offer cashback on Ewallet transactions to sustain its dominance
4. Stock peak categories ahead of January for maximum revenue capture
5. Improve service quality for the lowest-rated product line

---
*Built with Python · pandas · plotly · seaborn · Streamlit*